# Module 3 • Classical Natural Language Processing

# Lesson 20 • Classical NLP Capstone — Building and Evaluating an End-to-End Text Analytics Pipeline

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 150–180 minutes

---

## Scope

This capstone integrates the principal classical NLP components developed in
the previous lessons:

- text inspection and preprocessing;
- tokenization and normalization;
- TF-IDF representation;
- supervised text classification;
- information retrieval;
- topic modeling;
- rule-based entity extraction;
- evaluation and error analysis;
- reproducible artifact export.

The notebook uses a self-contained synthetic customer-support corpus so that
every cell can run without external downloads.

## Learning Objectives

After completing this lesson, the learner should be able to:

- define an end-to-end NLP problem and its outputs;
- inspect a labeled text dataset before modeling;
- design deterministic preprocessing that preserves task-relevant information;
- build leakage-safe classification pipelines;
- compare classical classifiers using stratified cross-validation;
- evaluate a final model on held-out data;
- interpret confusion matrices and influential features;
- construct TF-IDF document retrieval;
- calculate retrieval metrics;
- train and interpret an LDA topic model;
- add a rule-based entity-extraction component;
- combine multiple NLP outputs in one analysis function;
- document assumptions, limitations, and reproducibility metadata;
- export model outputs and reports.

## Table of Contents

1. Capstone Problem Definition
2. System Architecture
3. Dataset Construction
4. Data Validation
5. Exploratory Analysis
6. Text Preprocessing
7. Train-Test Split
8. Classification Baselines
9. Cross-Validated Model Comparison
10. Final Classification Model
11. Classification Evaluation
12. Feature Interpretation
13. Classification Error Analysis
14. Information Retrieval Component
15. Retrieval Evaluation
16. Topic Modeling Component
17. Topic Interpretation
18. Rule-Based Entity Extraction
19. Unified Text-Analytics Function
20. Batch Analysis
21. Confidence and Rejection
22. Reproducibility Metadata
23. Artifact Export
24. System Limitations
25. Knowledge Check
26. Capstone Extensions
27. Summary and Next Module

# 1. Capstone Problem Definition

We will build a customer-support text analytics system with four outputs:

1. **Classification:** predict the support category.
2. **Retrieval:** rank relevant knowledge-base articles.
3. **Topic modeling:** discover themes in the ticket collection.
4. **Entity extraction:** identify order IDs, dates, email addresses, and
   monetary expressions.

Supported classification labels:

- `account`;
- `billing`;
- `delivery`;
- `technical`.

## 1.1 Input and Output Contract

Input:

```text
One customer-support message
```

Output:

```text
predicted category
class confidence
retrieved support articles
extracted entities
dominant topic distribution
```

# 2. System Architecture

```text
Raw ticket
    │
    ├── Preprocessing
    │       ├── Unicode cleanup
    │       ├── whitespace normalization
    │       └── task-aware token preservation
    │
    ├── Classification pipeline
    │       ├── TF-IDF
    │       └── Logistic Regression
    │
    ├── Retrieval pipeline
    │       ├── TF-IDF knowledge-base index
    │       └── cosine similarity
    │
    ├── Entity extraction
    │       └── regular-expression rules
    │
    └── Topic model
            ├── count vectorizer
            └── LDA
```

Each component has a separate objective and evaluation method. Combining them
does not remove the need to evaluate them individually.

# 3. Dataset Construction

The synthetic dataset contains varied wording, identifiers, and dates.

In [ ]:
import pandas as pd

tickets = pd.DataFrame(
    [
        ("T001", "I forgot my password and cannot sign in", "account"),
        ("T002", "The login verification code does not arrive", "account"),
        ("T003", "Please change the email linked to my profile", "account"),
        ("T004", "My account has been locked after several attempts", "account"),
        ("T005", "I cannot access the security settings", "account"),
        ("T006", "How do I update my password?", "account"),
        ("T007", "The sign-in page rejects my credentials", "account"),
        ("T008", "Please delete my user profile", "account"),
        ("T009", "My username is not recognized", "account"),
        ("T010", "I need to verify my new email address", "account"),

        ("T011", "I was charged twice for order ORD-1001", "billing"),
        ("T012", "Please email the invoice to sara@example.com", "billing"),
        ("T013", "I need a refund for the payment made on 2026-07-20", "billing"),
        ("T014", "Why did the subscription price increase to $35?", "billing"),
        ("T015", "My card payment was declined", "billing"),
        ("T016", "The receipt shows an incorrect amount", "billing"),
        ("T017", "Please cancel the paid subscription", "billing"),
        ("T018", "Where can I download last month's invoice?", "billing"),
        ("T019", "A $19.99 charge appeared unexpectedly", "billing"),
        ("T020", "The renewal payment failed yesterday", "billing"),

        ("T021", "Order ORD-2001 has not arrived", "delivery"),
        ("T022", "The package is delayed beyond the delivery date", "delivery"),
        ("T023", "Tracking number TRK-4432 shows no movement", "delivery"),
        ("T024", "The courier delivered my parcel to the wrong address", "delivery"),
        ("T025", "When will order ORD-3005 be shipped?", "delivery"),
        ("T026", "My package arrived damaged", "delivery"),
        ("T027", "The delivery status has not changed since Monday", "delivery"),
        ("T028", "I received only one item from the shipment", "delivery"),
        ("T029", "The courier could not find my address", "delivery"),
        ("T030", "Please reschedule delivery for 2026-07-30", "delivery"),

        ("T031", "The application crashes during startup", "technical"),
        ("T032", "The dashboard loads very slowly", "technical"),
        ("T033", "File upload fails with server error 500", "technical"),
        ("T034", "The mobile app freezes after login", "technical"),
        ("T035", "Notifications are not being delivered", "technical"),
        ("T036", "The integration stopped working today", "technical"),
        ("T037", "The page displays a blank screen", "technical"),
        ("T038", "I cannot install the latest update", "technical"),
        ("T039", "The system logs me out repeatedly", "technical"),
        ("T040", "The report export feature does not work", "technical"),
    ],
    columns=["ticket_id", "text", "label"],
)

tickets.head()

In [ ]:
print("Rows:", len(tickets))
print("\nClass distribution:")
print(tickets["label"].value_counts())

# 4. Data Validation

Before modeling, validate:

- required columns;
- missing values;
- duplicate IDs;
- duplicate text;
- allowed labels;
- empty messages.

In [ ]:
REQUIRED_COLUMNS = {"ticket_id", "text", "label"}
ALLOWED_LABELS = {"account", "billing", "delivery", "technical"}

missing_columns = REQUIRED_COLUMNS - set(tickets.columns)
duplicate_ids = tickets["ticket_id"].duplicated().sum()
duplicate_texts = tickets["text"].duplicated().sum()
missing_values = tickets[list(REQUIRED_COLUMNS)].isna().sum().sum()
invalid_labels = set(tickets["label"]) - ALLOWED_LABELS
empty_texts = tickets["text"].str.strip().eq("").sum()

validation_report = pd.Series(
    {
        "missing_columns": sorted(missing_columns),
        "duplicate_ids": int(duplicate_ids),
        "duplicate_texts": int(duplicate_texts),
        "missing_values": int(missing_values),
        "invalid_labels": sorted(invalid_labels),
        "empty_texts": int(empty_texts),
    },
    name="Validation result",
)

validation_report

In [ ]:
assert not missing_columns
assert duplicate_ids == 0
assert missing_values == 0
assert not invalid_labels
assert empty_texts == 0

print("Dataset validation passed.")

# 5. Exploratory Analysis

Inspect document length, class balance, and recurring terms before building a
model.

In [ ]:
tickets["word_count"] = tickets["text"].str.split().str.len()
tickets["character_count"] = tickets["text"].str.len()

tickets.groupby("label")[
    ["word_count", "character_count"]
].agg(["mean", "min", "max"]).round(2)

In [ ]:
from collections import Counter
import re

WORD_PATTERN = re.compile(r"\b\w+(?:[-']\w+)*\b", flags=re.UNICODE)

def simple_tokens(text: str) -> list[str]:
    return WORD_PATTERN.findall(text.lower())


token_counts = Counter(
    token
    for text in tickets["text"]
    for token in simple_tokens(text)
)

pd.DataFrame(
    token_counts.most_common(20),
    columns=["token", "count"],
)

High-frequency words may reflect useful class signals, generic support language,
or dataset artifacts.

# 6. Text Preprocessing

The classifier will use a deterministic preprocessing function that:

- normalizes Unicode;
- standardizes whitespace;
- lowercases text;
- replaces emails, order IDs, tracking IDs, dates, money, and error codes with
  placeholders.

Placeholders preserve information type while reducing identifier sparsity.

In [ ]:
import unicodedata

EMAIL_PATTERN = re.compile(
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
)
ORDER_PATTERN = re.compile(r"\bORD-\d+\b", flags=re.IGNORECASE)
TRACKING_PATTERN = re.compile(r"\bTRK-\d+\b", flags=re.IGNORECASE)
DATE_PATTERN = re.compile(r"\b\d{4}-\d{2}-\d{2}\b")
MONEY_PATTERN = re.compile(r"\$\d+(?:\.\d{1,2})?")
ERROR_CODE_PATTERN = re.compile(r"\b(?:error\s*)?\d{3}\b", flags=re.IGNORECASE)
WHITESPACE_PATTERN = re.compile(r"\s+")


def preprocess_text(text: str) -> str:
    normalized = unicodedata.normalize("NFKC", str(text))
    normalized = EMAIL_PATTERN.sub(" EMAIL_TOKEN ", normalized)
    normalized = ORDER_PATTERN.sub(" ORDER_TOKEN ", normalized)
    normalized = TRACKING_PATTERN.sub(" TRACKING_TOKEN ", normalized)
    normalized = DATE_PATTERN.sub(" DATE_TOKEN ", normalized)
    normalized = MONEY_PATTERN.sub(" MONEY_TOKEN ", normalized)
    normalized = ERROR_CODE_PATTERN.sub(" ERROR_TOKEN ", normalized)
    normalized = normalized.lower()
    normalized = WHITESPACE_PATTERN.sub(" ", normalized).strip()
    return normalized

In [ ]:
preprocessing_examples = tickets.loc[
    [10, 11, 12, 13, 22, 32],
    ["text"],
].copy()

preprocessing_examples["processed"] = (
    preprocessing_examples["text"].map(preprocess_text)
)

preprocessing_examples

Placeholders should be selected according to the task. Replacing every number
with one generic token may erase distinctions between dates, prices, and error
codes.

# 7. Train-Test Split

The raw text is split before vectorizer fitting.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    tickets["text"],
    tickets["label"],
    test_size=0.25,
    random_state=42,
    stratify=tickets["label"],
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("\nTest distribution:")
print(y_test.value_counts())

# 8. Classification Baselines

We compare:

- Multinomial Naive Bayes;
- Logistic Regression;
- Linear Support Vector Machine.

Each model is placed inside a leakage-safe pipeline.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


def make_vectorizer() -> TfidfVectorizer:
    return TfidfVectorizer(
        preprocessor=preprocess_text,
        token_pattern=r"(?u)\b\w[\w_-]*\b",
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
    )


model_pipelines = {
    "Multinomial Naive Bayes": Pipeline(
        [
            ("tfidf", make_vectorizer()),
            ("classifier", MultinomialNB(alpha=0.5)),
        ]
    ),
    "Logistic Regression": Pipeline(
        [
            ("tfidf", make_vectorizer()),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=42,
                ),
            ),
        ]
    ),
    "Linear SVM": Pipeline(
        [
            ("tfidf", make_vectorizer()),
            (
                "classifier",
                LinearSVC(
                    C=1.0,
                    random_state=42,
                ),
            ),
        ]
    ),
}

list(model_pipelines)

# 9. Cross-Validated Model Comparison

We compare models using stratified cross-validation and macro F1.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

comparison_rows = []

for model_name, pipeline in model_pipelines.items():
    scores = cross_val_score(
        pipeline,
        tickets["text"],
        tickets["label"],
        cv=cross_validation,
        scoring="f1_macro",
    )

    comparison_rows.append(
        {
            "model": model_name,
            "mean_macro_f1": scores.mean(),
            "std_macro_f1": scores.std(),
            "fold_scores": scores.round(3).tolist(),
        }
    )

model_comparison = pd.DataFrame(comparison_rows)
model_comparison.sort_values(
    "mean_macro_f1",
    ascending=False,
).reset_index(drop=True)

The dataset is small and synthetic. Cross-validation demonstrates the
evaluation procedure but does not estimate production performance.

# 10. Final Classification Model

Logistic Regression is selected because it supports probabilities and
interpretable class-specific coefficients.

In [ ]:
final_classifier = model_pipelines["Logistic Regression"]

final_classifier.fit(
    X_train,
    y_train,
)

test_predictions = final_classifier.predict(X_test)
test_probabilities = final_classifier.predict_proba(X_test)

print("Final classifier fitted.")

# 11. Classification Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

test_accuracy = accuracy_score(
    y_test,
    test_predictions,
)

test_macro_f1 = f1_score(
    y_test,
    test_predictions,
    average="macro",
)

print(f"Test accuracy: {test_accuracy:.3f}")
print(f"Test macro F1: {test_macro_f1:.3f}")
print()
print(
    classification_report(
        y_test,
        test_predictions,
        zero_division=0,
    )
)

In [ ]:
class_names = final_classifier.named_steps["classifier"].classes_

confusion = confusion_matrix(
    y_test,
    test_predictions,
    labels=class_names,
)

confusion_frame = pd.DataFrame(
    confusion,
    index=[f"actual_{label}" for label in class_names],
    columns=[f"predicted_{label}" for label in class_names],
)

confusion_frame

Per-class metrics and confusion patterns should be reviewed even when aggregate
performance is high.

# 12. Feature Interpretation

Logistic Regression coefficients identify features associated with each class
in this fitted dataset.

In [ ]:
vectorizer = final_classifier.named_steps["tfidf"]
classifier = final_classifier.named_steps["classifier"]

feature_names = vectorizer.get_feature_names_out()

top_feature_rows = []

for class_index, class_name in enumerate(classifier.classes_):
    coefficients = classifier.coef_[class_index]
    top_indices = coefficients.argsort()[-10:][::-1]

    for rank, feature_index in enumerate(top_indices, start=1):
        top_feature_rows.append(
            {
                "class": class_name,
                "rank": rank,
                "feature": feature_names[feature_index],
                "coefficient": coefficients[feature_index],
            }
        )

top_features = pd.DataFrame(top_feature_rows)

top_features.head(12)

In [ ]:
top_features.pivot(
    index="rank",
    columns="class",
    values="feature",
)

Coefficients explain the fitted linear model, not universal meanings of words.

# 13. Classification Error Analysis

In [ ]:
test_results = pd.DataFrame(
    {
        "text": X_test.reset_index(drop=True),
        "actual": y_test.reset_index(drop=True),
        "predicted": test_predictions,
        "confidence": test_probabilities.max(axis=1),
    }
)

test_results["correct"] = (
    test_results["actual"]
    == test_results["predicted"]
)

test_results.sort_values(
    ["correct", "confidence"],
    ascending=[True, True],
)

In [ ]:
classification_errors = test_results[
    ~test_results["correct"]
].copy()

classification_errors

Error categories to inspect:

- label overlap;
- ambiguous wording;
- missing context;
- rare phrasing;
- preprocessing damage;
- mislabeled examples;
- domain shift.

# 14. Information Retrieval Component

We build a small knowledge base and rank articles using TF-IDF cosine
similarity.

In [ ]:
knowledge_base = pd.DataFrame(
    [
        (
            "KB01",
            "Reset Password",
            "Use the account security page to reset a forgotten password. "
            "A verification message is sent to the registered email.",
        ),
        (
            "KB02",
            "Change Account Email",
            "Open profile settings and confirm the current password before "
            "changing the registered email address.",
        ),
        (
            "KB03",
            "Download Invoice",
            "Invoices and receipts are available from the billing history page.",
        ),
        (
            "KB04",
            "Request Refund",
            "Eligible payments can be submitted for refund review from billing support.",
        ),
        (
            "KB05",
            "Track Delivery",
            "Use the order or tracking number to view shipment progress and delivery status.",
        ),
        (
            "KB06",
            "Report Damaged Package",
            "Photograph the damaged parcel and contact delivery support with the order ID.",
        ),
        (
            "KB07",
            "Resolve Application Crash",
            "Clear the application cache, restart the device, and install the latest version.",
        ),
        (
            "KB08",
            "Resolve Upload Error",
            "Check file size, file format, network connection, and server status.",
        ),
    ],
    columns=["article_id", "title", "body"],
)

knowledge_base["search_text"] = (
    knowledge_base["title"]
    + ". "
    + knowledge_base["body"]
)

knowledge_base

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

retrieval_vectorizer = TfidfVectorizer(
    preprocessor=preprocess_text,
    token_pattern=r"(?u)\b\w[\w_-]*\b",
    ngram_range=(1, 2),
    sublinear_tf=True,
)

knowledge_matrix = retrieval_vectorizer.fit_transform(
    knowledge_base["search_text"]
)

print("Knowledge-base matrix shape:", knowledge_matrix.shape)

In [ ]:
def retrieve_articles(
    query: str,
    top_k: int = 3,
    minimum_score: float = 0.0,
) -> pd.DataFrame:
    if top_k <= 0:
        raise ValueError("top_k must be positive")

    query_vector = retrieval_vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        knowledge_matrix,
    ).ravel()

    results = knowledge_base[
        ["article_id", "title", "body"]
    ].copy()

    results["score"] = scores

    return (
        results[results["score"] >= minimum_score]
        .sort_values(
            ["score", "article_id"],
            ascending=[False, True],
        )
        .head(top_k)
        .reset_index(drop=True)
    )


retrieve_articles(
    "My package arrived damaged",
    top_k=3,
)

# 15. Retrieval Evaluation

Relevance judgments define which articles are relevant to selected queries.

In [ ]:
retrieval_judgments = {
    "forgot my password": {"KB01"},
    "change account email": {"KB02"},
    "download invoice receipt": {"KB03"},
    "request payment refund": {"KB04"},
    "where is my package": {"KB05"},
    "damaged delivery": {"KB06"},
    "application crashes": {"KB07"},
    "file upload fails": {"KB08"},
}


def precision_at_k(
    ranked_ids: list[str],
    relevant_ids: set[str],
    k: int,
) -> float:
    selected = ranked_ids[:k]

    if not selected:
        return 0.0

    return sum(
        item in relevant_ids
        for item in selected
    ) / k


def reciprocal_rank(
    ranked_ids: list[str],
    relevant_ids: set[str],
) -> float:
    for rank, item in enumerate(ranked_ids, start=1):
        if item in relevant_ids:
            return 1.0 / rank

    return 0.0

In [ ]:
retrieval_rows = []

for query, relevant_ids in retrieval_judgments.items():
    ranked_ids = retrieve_articles(
        query,
        top_k=len(knowledge_base),
    )["article_id"].tolist()

    retrieval_rows.append(
        {
            "query": query,
            "precision_at_1": precision_at_k(
                ranked_ids,
                relevant_ids,
                1,
            ),
            "precision_at_3": precision_at_k(
                ranked_ids,
                relevant_ids,
                3,
            ),
            "reciprocal_rank": reciprocal_rank(
                ranked_ids,
                relevant_ids,
            ),
            "top_result": ranked_ids[0],
        }
    )

retrieval_evaluation = pd.DataFrame(retrieval_rows)

retrieval_evaluation

In [ ]:
retrieval_summary = pd.Series(
    {
        "Mean Precision@1": retrieval_evaluation["precision_at_1"].mean(),
        "Mean Precision@3": retrieval_evaluation["precision_at_3"].mean(),
        "MRR": retrieval_evaluation["reciprocal_rank"].mean(),
    },
    name="Retrieval metrics",
)

retrieval_summary.round(3)

# 16. Topic Modeling Component

LDA is applied to the complete ticket corpus for exploratory thematic analysis.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

topic_vectorizer = CountVectorizer(
    preprocessor=preprocess_text,
    token_pattern=r"(?u)\b\w[\w_-]*\b",
    stop_words="english",
    min_df=2,
    max_df=0.95,
)

topic_count_matrix = topic_vectorizer.fit_transform(
    tickets["text"]
)

topic_terms = topic_vectorizer.get_feature_names_out()

topic_model = LatentDirichletAllocation(
    n_components=4,
    learning_method="batch",
    max_iter=40,
    random_state=42,
)

ticket_topic_matrix = topic_model.fit_transform(
    topic_count_matrix
)

print("Topic matrix shape:", ticket_topic_matrix.shape)

# 17. Topic Interpretation

In [ ]:
def top_topic_terms(
    model: LatentDirichletAllocation,
    terms,
    top_n: int = 10,
) -> pd.DataFrame:
    rows = []

    for topic_index, weights in enumerate(model.components_):
        indices = weights.argsort()[-top_n:][::-1]

        rows.append(
            {
                "topic": f"Topic {topic_index}",
                "top_terms": ", ".join(
                    terms[index]
                    for index in indices
                ),
            }
        )

    return pd.DataFrame(rows)


topic_terms_frame = top_topic_terms(
    topic_model,
    topic_terms,
)

topic_terms_frame

In [ ]:
dominant_topics = ticket_topic_matrix.argmax(axis=1)
dominant_scores = ticket_topic_matrix.max(axis=1)

topic_assignment = tickets[
    ["ticket_id", "text", "label"]
].copy()

topic_assignment["dominant_topic"] = [
    f"Topic {index}"
    for index in dominant_topics
]

topic_assignment["topic_score"] = dominant_scores

topic_assignment.head()

In [ ]:
representative_documents = []

for topic_index in range(topic_model.n_components):
    best_index = ticket_topic_matrix[:, topic_index].argmax()

    representative_documents.append(
        {
            "topic": f"Topic {topic_index}",
            "ticket_id": tickets.iloc[best_index]["ticket_id"],
            "text": tickets.iloc[best_index]["text"],
            "score": ticket_topic_matrix[best_index, topic_index],
        }
    )

pd.DataFrame(representative_documents)

Topic labels should be assigned only after reviewing top terms and
representative tickets.

# 18. Rule-Based Entity Extraction

We extract:

- email addresses;
- order IDs;
- tracking IDs;
- ISO dates;
- monetary values;
- error codes.

In [ ]:
ENTITY_PATTERNS = {
    "EMAIL": EMAIL_PATTERN,
    "ORDER_ID": ORDER_PATTERN,
    "TRACKING_ID": TRACKING_PATTERN,
    "DATE": DATE_PATTERN,
    "MONEY": MONEY_PATTERN,
    "ERROR_CODE": ERROR_CODE_PATTERN,
}


def extract_entities(text: str) -> list[dict]:
    entities = []

    for entity_type, pattern in ENTITY_PATTERNS.items():
        for match in pattern.finditer(text):
            entities.append(
                {
                    "text": match.group(0),
                    "type": entity_type,
                    "start": match.start(),
                    "end": match.end(),
                }
            )

    return sorted(
        entities,
        key=lambda item: (
            item["start"],
            item["end"],
            item["type"],
        ),
    )


entity_examples = [
    "I was charged $19.99 for order ORD-1001 on 2026-07-20.",
    "Email the invoice to sara@example.com.",
    "Tracking number TRK-4432 has not moved.",
    "Upload fails with error 500.",
]

for example in entity_examples:
    print(example)
    print(extract_entities(example))
    print()

Regex extraction is transparent and precise for regular formats, but it does
not recognize general names, organizations, or ambiguous expressions.

# 19. Unified Text-Analytics Function

The following function combines all components.

In [ ]:
def analyze_ticket(
    text: str,
    retrieval_top_k: int = 3,
) -> dict:
    class_probabilities = final_classifier.predict_proba([text])[0]
    predicted_index = int(class_probabilities.argmax())
    predicted_label = classifier.classes_[predicted_index]
    confidence = float(class_probabilities[predicted_index])

    topic_input = topic_vectorizer.transform([text])
    topic_distribution = topic_model.transform(topic_input)[0]
    dominant_topic_index = int(topic_distribution.argmax())

    retrieved = retrieve_articles(
        text,
        top_k=retrieval_top_k,
    )

    return {
        "text": text,
        "processed_text": preprocess_text(text),
        "predicted_label": predicted_label,
        "classification_confidence": confidence,
        "class_probabilities": {
            class_name: float(probability)
            for class_name, probability in zip(
                classifier.classes_,
                class_probabilities,
            )
        },
        "dominant_topic": f"Topic {dominant_topic_index}",
        "topic_distribution": {
            f"Topic {index}": float(value)
            for index, value in enumerate(topic_distribution)
        },
        "entities": extract_entities(text),
        "retrieved_articles": retrieved[
            ["article_id", "title", "score"]
        ].to_dict(orient="records"),
    }

In [ ]:
sample_analysis = analyze_ticket(
    "Order ORD-9001 arrived damaged and I need a refund of $49.99"
)

sample_analysis

A unified function simplifies application integration, but each component still
requires independent monitoring and evaluation.

# 20. Batch Analysis

In [ ]:
new_tickets = [
    "I forgot my login password",
    "Please send the invoice to user@example.com",
    "Package ORD-5004 has not arrived",
    "The app crashes with error 503",
]

batch_rows = []

for text in new_tickets:
    result = analyze_ticket(
        text,
        retrieval_top_k=1,
    )

    batch_rows.append(
        {
            "text": text,
            "predicted_label": result["predicted_label"],
            "confidence": result["classification_confidence"],
            "dominant_topic": result["dominant_topic"],
            "top_article": (
                result["retrieved_articles"][0]["article_id"]
                if result["retrieved_articles"]
                else None
            ),
            "entity_count": len(result["entities"]),
        }
    )

batch_analysis = pd.DataFrame(batch_rows)
batch_analysis

# 21. Confidence and Rejection

A production system may abstain when classification confidence is below a
selected threshold.

In [ ]:
def classify_with_rejection(
    text: str,
    threshold: float = 0.55,
) -> dict:
    probabilities = final_classifier.predict_proba([text])[0]
    best_index = int(probabilities.argmax())
    label = classifier.classes_[best_index]
    confidence = float(probabilities[best_index])

    return {
        "label": label if confidence >= threshold else "REVIEW",
        "confidence": confidence,
        "threshold": threshold,
    }


rejection_examples = [
    "I cannot sign in",
    "Something is wrong",
    "The parcel arrived late",
]

for text in rejection_examples:
    print(text, "->", classify_with_rejection(text))

The threshold should be selected using validation data and application costs.
Confidence from Logistic Regression is not automatically well calibrated.

# 22. Reproducibility Metadata

In [ ]:
import platform
import sklearn

reproducibility_metadata = {
    "course": "Natural Language Processing: From Foundations to Large Language Models",
    "lesson": "Lesson 20 • Classical NLP Capstone — Building and Evaluating an End-to-End Text Analytics Pipeline",
    "author": "Eman Khater",
    "random_state": 42,
    "ticket_count": int(len(tickets)),
    "label_count": int(tickets["label"].nunique()),
    "classifier": "TF-IDF + Logistic Regression",
    "retriever": "TF-IDF + cosine similarity",
    "topic_model": "LDA with 4 topics",
    "entity_extraction": "regular expressions",
    "python_version": platform.python_version(),
    "scikit_learn_version": sklearn.__version__,
    "pandas_version": pd.__version__,
}

pd.Series(
    reproducibility_metadata,
    name="Value",
)

Reproducibility also requires versioned data, preprocessing code, and evaluation
judgments.

# 23. Artifact Export

The notebook exports:

- classification test predictions;
- model comparison results;
- retrieval evaluation;
- topic assignments;
- reproducibility metadata;
- one sample unified analysis.

In [ ]:
from pathlib import Path
import json

ARTIFACT_DIR = Path("lesson_20_outputs")
ARTIFACT_DIR.mkdir(exist_ok=True)

test_results.to_csv(
    ARTIFACT_DIR / "classification_test_results.csv",
    index=False,
)

model_comparison.to_csv(
    ARTIFACT_DIR / "model_comparison.csv",
    index=False,
)

retrieval_evaluation.to_csv(
    ARTIFACT_DIR / "retrieval_evaluation.csv",
    index=False,
)

topic_assignment.to_csv(
    ARTIFACT_DIR / "topic_assignments.csv",
    index=False,
)

with open(
    ARTIFACT_DIR / "reproducibility_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        reproducibility_metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

with open(
    ARTIFACT_DIR / "sample_analysis.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        sample_analysis,
        file,
        ensure_ascii=False,
        indent=2,
    )

sorted(path.name for path in ARTIFACT_DIR.iterdir())

Exported artifacts make experiments easier to inspect, compare, and reproduce.
Model serialization is intentionally omitted because this lesson focuses on the
analysis workflow.

# 24. System Limitations

This capstone has several limitations:

- synthetic and small data;
- limited linguistic variation;
- no probability calibration;
- lexical rather than semantic retrieval;
- no learned NER model;
- topic interpretation remains manual;
- no external validation set;
- no latency or memory benchmark;
- no production drift evaluation;
- English-only classification data.

## 24.1 Common Integration Risks

- using different preprocessing across components;
- fitting vectorizers on evaluation data;
- returning low-score retrieval results;
- treating topic labels as ground truth;
- treating regex matches as complete NER;
- over-trusting model confidence;
- exporting personally identifiable information;
- failing to version artifacts.

In [ ]:
integration_risks = pd.DataFrame(
    [
        ("Inconsistent preprocessing", "features do not align"),
        ("Evaluation leakage", "inflated performance"),
        ("Low-score retrieval", "irrelevant recommendations"),
        ("Overconfident automation", "incorrect routing"),
        ("Unversioned artifacts", "irreproducible results"),
        ("Unredacted exports", "privacy risk"),
    ],
    columns=["Risk", "Consequence"],
)

integration_risks

# 25. Knowledge Check

1. Why should an NLP system define an input-output contract?
2. Which validation checks should occur before modeling?
3. Why are placeholders useful for structured identifiers?
4. Why must vectorizers remain inside classification pipelines?
5. Why compare models with the same cross-validation folds?
6. Why inspect per-class metrics?
7. What do Logistic Regression coefficients represent?
8. Why must a retrieval query use the fitted document vectorizer?
9. What does MRR measure?
10. Why should LDA topics be interpreted manually?
11. Which entity types are suitable for regex extraction?
12. Why does a unified function not replace component evaluation?
13. Why might a system reject low-confidence predictions?
14. Which artifacts should be versioned?
15. What are the major limitations of this capstone?

# 26. Capstone Extensions

## Exercise 1 — Real Dataset

Replace the synthetic tickets with a public or private dataset and document the
label definitions.

## Exercise 2 — Hyperparameter Tuning

Tune n-gram range, `min_df`, Logistic Regression `C`, and class weights.

## Exercise 3 — Probability Calibration

Add calibration and assess reliability with calibration curves and Brier score.

## Exercise 4 — Improved Retrieval

Add BM25 or semantic embeddings and compare ranking metrics.

## Exercise 5 — Learned NER

Replace regex-only extraction with a sequence-labeling model.

## Exercise 6 — Arabic Pipeline

Build an Arabic support-ticket version with language-specific normalization.

## Exercise 7 — Monitoring

Add class-distribution, vocabulary-coverage, confidence, and zero-result
monitoring.

## Exercise 8 — Packaging

Refactor the notebook into reusable Python modules and command-line scripts.

## Challenge Project

Build a reproducible text analytics package containing:

- data validation;
- preprocessing;
- training;
- cross-validation;
- final evaluation;
- retrieval;
- topic modeling;
- entity extraction;
- batch inference;
- artifact export;
- tests;
- documentation.

# 27. Summary and Next Module

In this capstone:

- a complete NLP problem and output contract were defined;
- the dataset was validated and explored;
- deterministic preprocessing preserved structured information;
- TF-IDF classifiers were compared with stratified cross-validation;
- a final Logistic Regression model was evaluated and interpreted;
- classification errors were organized for review;
- TF-IDF cosine retrieval ranked knowledge-base articles;
- retrieval quality was measured with Precision@k and MRR;
- LDA discovered latent themes in the ticket corpus;
- regex rules extracted structured entities;
- one function combined classification, retrieval, topics, and entities;
- confidence rejection and reproducibility metadata were added;
- analysis artifacts were exported for inspection.

## Next Module

**Module 4: Distributional Semantics and Word Embeddings**

**Lesson 21: Distributional Semantics and Word Embedding Foundations** moves
from sparse lexical features to dense representations, similarity in embedding
spaces, and the distributional hypothesis.

# References

- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.
- Manning, C. D., Raghavan, P., & Schütze, H. *Introduction to Information Retrieval*.
- scikit-learn text feature extraction, classification, and LDA documentation.
- classical NLP evaluation and reproducible machine-learning literature.